# Dyad Learning Experiment Overview

## Idea
This experiment tests whether collaborative dyad learning improves reinforcement learning performance compared to single-agent training. The environment is the PUZZLES benchmark (Simon Tatham's Portable Puzzle Collection) wrapped as a Gymnasium environment. Agents are trained on the Netslide puzzle with two parameter settings and evaluated on return, success rate, and episode length.

## Setup
- Environment: PUZZLES (Gymnasium + Pygame + C backend)
- Puzzle: Netslide
- Puzzle params: 2x3b1 and 3x3b1
- Observation modes: discrete internal game state or RGB pixels
- Algorithm: DQN with replay buffer and target network
- Training: epsilon-greedy policy, action masking optional
- Evaluation: periodic runs with fixed seeds and logged metrics
- Outputs: checkpoints, logs, results, and visualizations saved under results/ and logs/

## Experiments
1. Baseline DQN with discrete state input (MLP).
2. DQN with RGB pixel input (CNN).
3. Dyad learning: two agents periodically evaluate, exchange experiences (both obs types stored), rate each other's trajectories using their own value function, filter by ratings, and merge into both replay buffers.

## Agent Variants (Detailed)

### Agent A: Discrete State DQN (MLP)
- **Input**: puzzle internal state dict (arrays + scalars) from `info["puzzle_state"]`.
- **Preprocessing**: flatten and concatenate all numeric fields into a single vector; normalize to [0, 1] where possible.
- **Network**: MLP with 2-3 hidden layers (ReLU), output size = number of discrete actions.
- **Strengths**: direct access to logical state; faster learning and better sample efficiency.
- **Risk**: puzzle-specific input schema requires careful flattening and consistent ordering.

### Agent B: Pixel DQN (CNN)
- **Input**: RGB observation array with shape (3, W, H) and values in [0, 255].
- **Preprocessing**: normalize to [0, 1]; optional frame stacking if needed for dynamics.
- **Network**: CNN feature extractor + MLP head for Q-values.
- **Strengths**: consistent input across puzzles; less engineering for per-puzzle state parsing.
- **Risk**: harder exploration; higher sample complexity; may need more training steps.

## Dyad Learning: Experience Sharing Protocol
- **When**: after every `n` training episodes, each agent runs one evaluation episode.
- **What is stored**: for every transition, store both observation modalities (RGB + discrete state) plus action, reward, next observations, and termination flags.
- **Rating**: the other agent computes expected return from its own value function on those transitions and compares to observed return.
- **Filtering**: keep transitions above a configurable rating threshold; discard low-quality trajectories.
- **Merge**: combine accepted transitions from both agents and add to both replay buffers.
- **Goal**: transfer useful trajectories across input modalities to improve convergence and generalization.

In [1]:
import gymnasium as gym
import rlp
import pygame

In [ ]:
# render_mode = "human"
render_mode = "rgb_array"
env = gym.make('rlp/Puzzle-v0', puzzle="mines",
               render_mode=render_mode, params="4x4n2",
               window_width=512, window_height=512,
               obs_type='puzzle_state',
               include_cursor_in_state_info=True
               )

observation, info = env.reset(seed=42)
for _ in range(10):
    action = env.action_space.sample()  # currently a random policy
    observation, reward, terminated, truncated, info = env.step(action)
    print(
        f"Observation: {observation},\nAction: {action},\nReward: {reward},\nInfo: {info}\n")

    if terminated or truncated:
        observation, info = env.reset()
        print(f"Episode complete, resetting")
env.close()

Observation: {'w': 4, 'h': 4, 'n': 2, 'dead': 0, 'unique': 1, 'mines': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8), 'grid': array([-2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2],
      dtype=int8), 'cursor_pos': array([0, 0], dtype=int32)},
Action: 0,
Reward: 0,
Info: {'puzzle_state': {'w': 4, 'h': 4, 'n': 2, 'dead': False, 'won': False, 'used_solve': False, 'layout': {'mines': [], 'n': 2, 'unique': True}, 'grid': [-2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2], 'cursor_pos': (0, 0)}, 'state_histogram': {108626919667501783: 2}, 'current_state_repeats': 2, 'current_move_was_toward_solution': False, 'current_move_was_cursor_move': False}

Observation: {'w': 4, 'h': 4, 'n': 2, 'dead': 0, 'unique': 1, 'mines': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int8), 'grid': array([-2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2],
      dtype=int8), 'cursor_pos': array([0, 0], dtype=int32)},
Action: 2,
Rewa

: 